# Imports and basic settings

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import scipy.io
from scipy.stats import qmc
import matplotlib.pyplot as plt

torch.manual_seed(1234)
np.random.seed(1234)
torch.set_default_dtype(torch.float64)
os.makedirs("results", exist_ok=True)

# Load the reference solution

In [3]:
data = scipy.io.loadmat("data/burgers_shock.mat")
t_grid = data["t"].flatten()[:, None]
x_grid = data["x"].flatten()[:, None]
Exact = data["usol"]            # shape (256, 100): Exact[i, j] = u(x_i, t_j)

X, T = np.meshgrid(x_grid, t_grid)
X_star = np.hstack((T.flatten()[:, None], X.flatten()[:, None]))
u_star = Exact.T.flatten()[:, None]

lb = X_star.min(0)   # [t_min, x_min]
ub = X_star.max(0)   # [t_max, x_max]
print("t in", lb[0], "to", ub[0])
print("x in", lb[1], "to", ub[1])

t in 0.0 to 0.99
x in -1.0 to 1.0


## Pick the 100 training points (initial + boundary)

In [4]:
xx1 = np.hstack((X[0:1, :].T, T[0:1, :].T))   # t = 0 row
uu1 = Exact[:, 0:1]
xx2 = np.hstack((X[:, 0:1], T[:, 0:1]))       # x = -1 column
uu2 = Exact[0:1, :].T
xx3 = np.hstack((X[:, -1:], T[:, -1:]))       # x = 1 column
uu3 = Exact[-1:, :].T

X_u_train_all = np.vstack([xx1, xx2, xx3])[:, [1, 0]]   # reorder to (t, x)
u_train_all = np.vstack([uu1, uu2, uu3])

N_u = 100
idx = np.random.choice(X_u_train_all.shape[0], N_u, replace=False)
X_u_train = X_u_train_all[idx, :]
u_train = u_train_all[idx, :]
print(X_u_train.shape, u_train.shape)

(100, 2) (100, 1)


## Scatter 10,000 "physics-only" points

In [5]:
N_f = 10000
sampler = qmc.LatinHypercube(d=2, seed=1234)
sample = sampler.random(n=N_f)
X_f_train = lb + (ub - lb) * sample
X_f_train = np.vstack([X_f_train, X_u_train])
print(X_f_train.shape)

(10100, 2)
